In [1]:
import pandas as pd
from darts.timeseries import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel
from darts.dataprocessing.transformers import StaticCovariatesTransformer
import numpy as np
import torch
import matplotlib.pyplot as plt
import joblib
import os

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Loss Logger 

In [2]:
from pytorch_lightning.callbacks import Callback

class LossLogger(Callback):
    """
    A PyTorch Lightning callback to record training and validation losses 
    at the end of every epoch for custom plotting or analysis.
    """
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        # Retrieve train_loss from callback_metrics
        train_loss = trainer.callback_metrics.get("train_loss")
        
        if train_loss is not None:
            # detach() ensures we don't keep the computation graph in memory
            # cpu() ensures it works regardless of whether you're on GPU or CPU
            self.train_losses.append(float(train_loss.detach().cpu()))

    def on_validation_epoch_end(self, trainer, pl_module):
        # Retrieve val_loss from callback_metrics
        val_loss = trainer.callback_metrics.get("val_loss")
        
        if val_loss is not None:
            self.val_losses.append(float(val_loss.detach().cpu()))

### Add encoders

In [3]:
# Function to encode the year as a normalized value
def encode_year(idx):
  return (idx.year - 2000) / 50

def encode_days_in_month(index):
  return index.days_in_month.to_numpy().reshape(-1,1)

# Set up the add_encoders dictionary to specify how different time-related encoders and transformers should be applied
add_encoders = {
    'cyclic': {'past': ['month'], 'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {
        'past': [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

### Read the data

In [4]:
pandas_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Input_Data_Preparation\Series_A_data.csv",index_col = ['MONTH_OF_SALE'],parse_dates=True)

In [5]:
print(f"Minimum date in the data is {pandas_df.index.min()}")

print(f"Maximum date in the data is {pandas_df.index.max()}")

Minimum date in the data is 2023-04-01 00:00:00
Maximum date in the data is 2026-11-01 00:00:00


### Updating marriage days

In [6]:
#March 2026 marriage days updated from 8 to 12
pandas_df.loc['2026-03-01','MARRIAGE_DAYS'] = 12

#April 2026 remains unchanged although 4 of Marriage days shifted to March 
# but 4 of May marriage days are also shifted to April

#May 2026 changed from 5 to 4
pandas_df.loc['2026-05-01','MARRIAGE_DAYS'] = 4

#June 2026 changed from 8 to 10 
pandas_df.loc['2026-06-01','MARRIAGE_DAYS'] = 10 

#July 2026 changed from 2 to 3
pandas_df.loc['2026-07-01','MARRIAGE_DAYS'] = 3

#November 2026 changed from 4 to 9
pandas_df.loc['2026-11-01','MARRIAGE_DAYS'] = 9




### Separating the type of variables

In [7]:
#Extracting type of columns according to the datatypes
# 1. Targets/Metrics (The numbers we want to predict)
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')
target_cols = list(set(target_cols))

# 2. Time Dimension
time_cols = pandas_df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()
time_cols = list(set(time_cols))

# 3. Static/Categorical Covariates (The identifiers)
# We exclude numbers and dates to find the "ID" strings
static_cols = pandas_df.select_dtypes(exclude=['number', 'datetime', 'datetime64']).columns.tolist()
static_cols.append('PARENT_DEALER_CODE')
static_cols.extend(['PARENT_DEALER_CODE','AKSHAYA_TRITIYA_DAYS', 'BUDDHA_PURNIMA_DAYS', 'EID_UL_FITR_DAYS', 'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GURU_PURNIMA_DAYS', 'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS', 'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS', 'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'NAG_PANCHAMI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS', 'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS', 'VISHWAKARMA_PUJA_DAYS','D-30', 'D-29', 'D-28', 'D-27', 'D-26', 'D-25', 'D-24', 'D-23', 'D-22', 'D-21', 'D-20', 'D-19', 'D-18', 'D-17', 'D-16', 'D-15', 'D-14', 'D-13', 'D-12', 'D-11', 'D-10', 'D-9', 'D-8', 'D-7', 'D-6', 'D-5', 'D-4', 'D-3', 'D-2', 'D-1', 'D0', 'D+1', 'D+2', 'D+3', 'D+4', 'D+5', 'D+6', 'D+7'])
static_cols = list(set(static_cols))

print(f"Targets: {target_cols}")
print(f"Time Column: {time_cols}")
print(f"Static Identifiers: {static_cols}")

Targets: ['D-6', 'NET_SALES', 'D-23', 'D-13', 'D-4', 'HANUMAN_JAYANTI_DAYS', 'D-10', 'JAGANNATH_RATHYATRA_DAYS', 'D-14', 'HOLIKA_DAHAN_DAYS', 'D-9', 'D-22', 'HOLI_DAYS', 'D-2', 'FESTIVE_PHASE_II', 'D-28', 'D-11', 'D+3', 'D-15', 'GANGA_DUSSEHRA_DAYS', 'D-30', 'D-20', 'D+6', 'LAST_YEAR_CONTRIBUTION', 'AKSHAYA_TRITIYA_DAYS', 'MAHA_SHIVARATRI_DAYS', 'REPUBLIC_DAY_DAYS', 'MARRIAGE_DAYS', 'FESTIVE_PHASE_I', 'ONAM_DAYS', 'D0', 'PARENT_DEALER_CODE', 'BUDDHA_PURNIMA_DAYS', 'D-1', 'D+1', 'HARTALIK_TEEJ_DAYS', 'D+7', 'EID_UL_FITR_DAYS', 'PROP_FESTIVE_PHASE_II', 'NEW_YEAR_DAYS', 'D-26', 'D-17', 'D-8', 'FESTIVE_PHASE_III', 'D-5', 'D-12', 'JANMASHTAMI_DAYS', 'D-29', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'PITRU_PAKSH', 'D-27', 'D-24', 'PARENT_DEALER_CODE_MODEL_FAMILY', 'D-3', 'D+5', 'D-25', 'NAG_PANCHAMI_DAYS', 'D+2', 'LOHRI_DAYS', 'VASANT_PANCHAMI_DAYS', 'GANESH_CHATURTHI_DAYS', 'GURU_PURNIMA_DAYS', 'PROP_PITRU_PAKSH', 'D-18', 'RAKSHA_BANDHAN_DAYS', 'PROP_FESTIVE_PHASE_III', 'D-7', 'D-16', 'D-19', 'PROP_FE

In [8]:
#Separating the covariates
target_col = ["NET_SALES"]

#future covariates
future_covariates = [i for i in target_cols if i not in ['NET_SALES','PARENT_DEALER_CODE','AKSHAYA_TRITIYA_DAYS', 'BUDDHA_PURNIMA_DAYS', 'EID_UL_FITR_DAYS', 'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GURU_PURNIMA_DAYS', 'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS', 'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS', 'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'NAG_PANCHAMI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS', 'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS', 'VISHWAKARMA_PUJA_DAYS','D-30', 'D-29', 'D-28', 'D-27', 'D-26', 'D-25', 'D-24', 'D-23', 'D-22', 'D-21', 'D-20', 'D-19', 'D-18', 'D-17', 'D-16', 'D-15', 'D-14', 'D-13', 'D-12', 'D-11', 'D-10', 'D-9', 'D-8', 'D-7', 'D-6', 'D-5', 'D-4', 'D-3', 'D-2', 'D-1', 'D0', 'D+1', 'D+2', 'D+3', 'D+4', 'D+5', 'D+6', 'D+7']]

#actual_static_cols
actual_static_cols = [i for i in static_cols if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

In [9]:
future_covariates

['FESTIVE_PHASE_II',
 'LAST_YEAR_CONTRIBUTION',
 'MARRIAGE_DAYS',
 'FESTIVE_PHASE_I',
 'PROP_FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'PROP_PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_III',
 'PROP_FESTIVE_PHASE_I']

In [10]:
actual_static_cols

['D-6',
 'D-23',
 'D-13',
 'COLOUR',
 'HANUMAN_JAYANTI_DAYS',
 'D-10',
 'JAGANNATH_RATHYATRA_DAYS',
 'D-14',
 'HOLIKA_DAHAN_DAYS',
 'D-9',
 'D-22',
 'HOLI_DAYS',
 'D-2',
 'D-28',
 'D-11',
 'D+3',
 'D-15',
 'GANGA_DUSSEHRA_DAYS',
 'D-30',
 'D-20',
 'DEALER_CITY',
 'D+6',
 'AKSHAYA_TRITIYA_DAYS',
 'X_CITY_CATEGORY',
 'MAHA_SHIVARATRI_DAYS',
 'REPUBLIC_DAY_DAYS',
 'ONAM_DAYS',
 'IGNITION_TYPE',
 'D0',
 'PARENT_DEALER_CODE',
 'BUDDHA_PURNIMA_DAYS',
 'D+1',
 'HARTALIK_TEEJ_DAYS',
 'D+7',
 'EID_UL_FITR_DAYS',
 'D-26',
 'NEW_YEAR_DAYS',
 'D-8',
 'D-17',
 'D-5',
 'D-4',
 'D-1',
 'D-12',
 'JANMASHTAMI_DAYS',
 'WHEEL_TYPE',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'D-29',
 'D-27',
 'ZONAL_OFFICE_NAME',
 'D-24',
 'D-3',
 'D+5',
 'D-25',
 'BRAKE_TYPE',
 'NAG_PANCHAMI_DAYS',
 'D+2',
 'MODEL_FAMILY',
 'LOHRI_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GURU_PURNIMA_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'D-18',
 'RAKSHA_BANDHAN_DAYS',
 'D-7',
 'D-16',
 'D-19',
 'D+4',
 'D-21',
 'VISHWAKARMA_PUJA_DAYS']

In [11]:
# #since variables like MODEL_FAMILY,BRAKE_VARIANT,IGNITION_TYPE,WHEEL_TYPE,BIKE_COLOUR are mostly same for all the top 10 series, will be removing them from the static covariates'
# static_covariates = [i for i in actual_static_cols if i not in ['MODEL_FAMILY','BRAKE_VARIANT','IGNITION_TYPE','WHEEL_TYPE','BIKE_COLOUR','DEALER_CODE']]
# static_covariates

static_covariates = actual_static_cols.copy()

static_covariates = list(set(static_covariates))

### Preparing data for Darts

In [12]:
#Step 1 - Sorting the dataframe by date
pandas_df=pandas_df.reset_index().sort_values(by=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"]).set_index("MONTH_OF_SALE")

In [13]:
#Step 2 - Separating the static covariates and NET_SALES column
pandas_df_with_target_and_static_covariates = pandas_df.loc[:,['PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']+static_covariates]
pandas_df_with_target_and_static_covariates.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES,D-6,D-23,D-13,HANUMAN_JAYANTI_DAYS,COLOUR,D-10,JAGANNATH_RATHYATRA_DAYS,D-14,...,GURU_PURNIMA_DAYS,VASANT_PANCHAMI_DAYS,D-18,RAKSHA_BANDHAN_DAYS,D-7,D-16,D-19,D+4,D-21,VISHWAKARMA_PUJA_DAYS
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2023-04-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,4.0,0.0,0.0,0.0,1,BLACK,0.0,0,0.0,...,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
2023-05-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,6.0,0.0,0.0,0.0,0,BLACK,0.0,0,0.0,...,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
2023-06-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,4.0,0.0,0.0,0.0,0,BLACK,0.0,1,0.0,...,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
2023-07-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,5.0,0.0,0.0,0.0,0,BLACK,0.0,0,0.0,...,1,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0
2023-08-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.0,0.0,0.0,0.0,0,BLACK,0.0,0,0.0,...,0,0,0.0,1,0.0,0.0,0.0,0.0,0.0,0


In [14]:
#Step 3 - Separating the future covariates
pandas_df_with_future_covariates = pandas_df.loc[:,future_covariates]
pandas_df_with_future_covariates.head()

,FESTIVE_PHASE_II,LAST_YEAR_CONTRIBUTION,MARRIAGE_DAYS,FESTIVE_PHASE_I,PROP_FESTIVE_PHASE_II,FESTIVE_PHASE_III,PITRU_PAKSH,PARENT_DEALER_CODE_MODEL_FAMILY,PROP_PITRU_PAKSH,PROP_FESTIVE_PHASE_III,PROP_FESTIVE_PHASE_I
MONTH_OF_SALE,,,,,,,,,,,
2023-04-01,0.0,0.0935,0.0,0.0,0.0,0.0,0.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,0.0,0.0
2023-05-01,0.0,0.1047,12.0,0.0,0.0,0.0,0.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,0.0,0.0
2023-06-01,0.0,0.0791,10.0,0.0,0.0,0.0,0.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,0.0,0.0
2023-07-01,0.0,0.0604,0.0,0.0,0.0,0.0,0.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,0.0,0.0
2023-08-01,0.0,0.0617,0.0,0.0,0.0,0.0,0.0,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,0.0,0.0


In [15]:
#Step 4 - Creating the darts timeseries object for target and static covariates
darts_df_with_static_covariates = TimeSeries.from_group_dataframe(df=pandas_df_with_target_and_static_covariates,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')


In [16]:
#Step 5 - Creating the darts timeseries object with future covariates

#Removing PARENT_DEALER_CODE_MODEL_FAMILY from future_covariates
try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass

darts_df_with_future_covariates = TimeSeries.from_group_dataframe(df = pandas_df_with_future_covariates,
                                    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                    freq = 'MS',
                                    value_cols = future_covariates
                                    )

### Train/Test split

In [17]:
#Step 6 - Creating train, test, and validation split
#Train set - Apr'23 to Dec'25 
#Val set - Jan'26 to Mar'26 


train_list = []
val_list = []

for ts in darts_df_with_static_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-05-01'))
    val = ts.slice(pd.Timestamp('2024-06-01'), pd.Timestamp('2026-05-01'))
    
    train_list.append(train)
    val_list.append(val)

train_future_covariates_list = []
validation_future_covariates_list = []

for ts in darts_df_with_future_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-05-01'))
    val = ts.slice(pd.Timestamp('2024-06-01'), pd.Timestamp('2026-05-01'))
    train_future_covariates_list.append(train)
    validation_future_covariates_list.append(val)

### Scaling

In [18]:

target_scaler = Scaler(n_jobs=-1)
future_covariates_scaler = Scaler(n_jobs=-1)

transformer = StaticCovariatesTransformer(n_jobs=-1)

#Scale the target training data
scaled_target_series = target_scaler.fit_transform(train_list)

scaled_target_series_with_static_covariates_training = transformer.fit_transform(scaled_target_series)



# #Scale the static covariates in training data
# scaled_static_covariates_training = transformer.fit_transform(train_list)

# #Scale the future covariates in training data
# # scaled_future_covariates = future_covariates_scaler.fit_transform(darts_df_with_future_covariates)

scaled_future_covariates_training = future_covariates_scaler.fit_transform(train_future_covariates_list)
scaled_future_covariates_validation = future_covariates_scaler.transform(validation_future_covariates_list)


# #Scale the target validation data
scaled_target_series_validation = target_scaler.transform(val_list)
scaled_target_series_with_static_covariates_validation = transformer.transform(scaled_target_series_validation)

# #Scale the static covariates in validation data
# scaled_static_covariates_validation = transformer.transform(val_list)



In [19]:
output_dir_scaler = r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling\output_directory_scaled_objects"
joblib.dump(target_scaler, os.path.join(output_dir_scaler, 'target_scaler.pkl'))
joblib.dump(future_covariates_scaler, os.path.join(output_dir_scaler, 'future_covariates_scaler.pkl'))
joblib.dump(transformer, os.path.join(output_dir_scaler, 'static_transformer.pkl'))

['C:\\Users\\G0004878\\Desktop\\TFT_Data\\6_month_horizon\\Updated_diwali_features\\Modelling_without_scaling\\output_directory_scaled_objects\\static_transformer.pkl']

In [20]:
from datetime import datetime

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from darts.models import TFTModel

In [21]:
from datetime import datetime
from pytorch_lightning.callbacks import ModelCheckpoint

now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")

WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling"
MODEL_NAME = f"tft_diwali_based_unscaled_binary_features_6_month_horizon_huber_loss_{now}"

MODEL_DIR = os.path.join(WORK_DIR, MODEL_NAME)
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

MODEL_DIR: C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling\tft_diwali_based_unscaled_binary_features_6_month_horizon_huber_loss_2026-06-26_18_17_56
CHECKPOINT_DIR: C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling\tft_diwali_based_unscaled_binary_features_6_month_horizon_huber_loss_2026-06-26_18_17_56\checkpoints


In [22]:
class DateStampedCheckpoint(ModelCheckpoint):
    @property
    def state_key(self) -> str:
        return f"DateStampedCheckpoint_{self.monitor}_{self.dirpath}"

In [23]:
checkpoint_callback = DateStampedCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="best_model",          # Completely static name: saves as best_model.ckpt
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=True,                 # Also retains 'last.ckpt' as a fallback backup
    verbose=True
)

In [24]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=25,
    mode="min",
    verbose=True
)

In [25]:
loss_logger = LossLogger()

model = TFTModel(
    input_chunk_length=16,
    output_chunk_length=6,
    batch_size=128,
    dropout=0.1,
    likelihood=None,
    loss_fn=torch.nn.HuberLoss(delta=0.1),
    n_epochs=200,
    random_state=42,
    add_encoders=add_encoders,
    model_name=MODEL_NAME,
    work_dir=WORK_DIR,
    
    # CRITICAL CHANGE: Tell Darts to handle its native model manifest building
    save_checkpoints=True,          
    force_reset=True,
    
    pl_trainer_kwargs={
        "callbacks": [
            loss_logger,
            early_stop_callback
        ],
        "enable_checkpointing": True,
        "gradient_clip_val": 0.1
    }
)

print("\nRunning LR Finder...")
lr_finder = model.lr_find(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training,
)

suggested_lr = lr_finder.suggestion()
print("Suggested Learning Rate:", suggested_lr)
model.lr = suggested_lr


Running LR Finder...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX 2000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at c:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling\.lr_find_00289697-fef7-4cb2-9b63-20080bf8d349.ckpt
Restored all states from the checkpoint at c:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling\.lr_find_00289697-fef7-4cb2-9b63-20080bf8d349.ckpt


Suggested Learning Rate: 0.000630957344480193


In [26]:
print(CHECKPOINT_DIR)

C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling\tft_diwali_based_unscaled_binary_features_6_month_horizon_huber_loss_2026-06-26_18_17_56\checkpoints


In [27]:
print("\nStarting Training with Validation...")
model.fit(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training,
    val_series=scaled_target_series_with_static_covariates_validation,
    val_future_covariates=scaled_future_covariates_validation,
    verbose=True
)

print(f"\n✅ Training Complete. Best model saved at:\n--> {os.path.join(CHECKPOINT_DIR, 'best_model.ckpt')}")


Starting Training with Validation...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | criterion                         | HuberLoss                        | 0      | train
1  | train_criterion                   | HuberLoss                        | 0      | train
2  | val_criterion                     | HuberLoss                        | 0      | train
3  | train_metrics                     | MetricCollection                 | 0      | train
4  | val_metrics                       | MetricCollection                 | 0      | train
5  | input_embeddings                  | _MultiEmbedding                  | 0      | train
6  | static_covariates_vsn             | _VariableSelectionNetwork        | 48.9 K | train
7  | encoder_vsn

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 25 records. Best score: 0.025. Signaling Trainer to stop.



✅ Training Complete. Best model saved at:
--> C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Modelling_without_scaling\tft_diwali_based_unscaled_binary_features_6_month_horizon_huber_loss_2026-06-26_18_17_56\checkpoints\best_model.ckpt


In [28]:
from darts.models import TFTModel

# Direct native restoration call
loaded_model = TFTModel.load_from_checkpoint(model_name='tft_diwali_based_unscaled_binary_features_6_month_horizon_huber_loss_2026-06-26_18_17_56',best=True,map_location="cpu")
print("Model loaded successfully with all dimensions intact!")

Model loaded successfully with all dimensions intact!


### Creating the prediction data

In [29]:
prediction_df = pandas_df.loc[((pandas_df.index>='2025-02-01') & (pandas_df.index<='2026-11-01'))]
prediction_df.shape

(794156, 81)

In [30]:
target_plus_static_cols = target_col + static_cols
target_plus_static_cols

['NET_SALES',
 'D-6',
 'D-23',
 'D-13',
 'COLOUR',
 'HANUMAN_JAYANTI_DAYS',
 'D-10',
 'JAGANNATH_RATHYATRA_DAYS',
 'D-14',
 'HOLIKA_DAHAN_DAYS',
 'D-9',
 'D-22',
 'HOLI_DAYS',
 'D-2',
 'D-28',
 'D-11',
 'D+3',
 'D-15',
 'GANGA_DUSSEHRA_DAYS',
 'D-30',
 'D-20',
 'DEALER_CITY',
 'D+6',
 'AKSHAYA_TRITIYA_DAYS',
 'X_CITY_CATEGORY',
 'MAHA_SHIVARATRI_DAYS',
 'REPUBLIC_DAY_DAYS',
 'ONAM_DAYS',
 'IGNITION_TYPE',
 'D0',
 'PARENT_DEALER_CODE',
 'BUDDHA_PURNIMA_DAYS',
 'D+1',
 'HARTALIK_TEEJ_DAYS',
 'D+7',
 'EID_UL_FITR_DAYS',
 'D-26',
 'NEW_YEAR_DAYS',
 'D-8',
 'D-17',
 'D-5',
 'D-4',
 'D-1',
 'D-12',
 'JANMASHTAMI_DAYS',
 'WHEEL_TYPE',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'D-29',
 'D-27',
 'ZONAL_OFFICE_NAME',
 'D-24',
 'D-3',
 'D+5',
 'D-25',
 'BRAKE_TYPE',
 'NAG_PANCHAMI_DAYS',
 'D+2',
 'MODEL_FAMILY',
 'LOHRI_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GURU_PURNIMA_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'D-18',
 'RAKSHA_BANDHAN_DAYS',
 'D-7',
 'D-16',
 'D-19',
 'D+4',
 'D-2

In [31]:
static_plus_future_cov = static_cols + future_covariates
static_plus_future_cov

['D-6',
 'D-23',
 'D-13',
 'COLOUR',
 'HANUMAN_JAYANTI_DAYS',
 'D-10',
 'JAGANNATH_RATHYATRA_DAYS',
 'D-14',
 'HOLIKA_DAHAN_DAYS',
 'D-9',
 'D-22',
 'HOLI_DAYS',
 'D-2',
 'D-28',
 'D-11',
 'D+3',
 'D-15',
 'GANGA_DUSSEHRA_DAYS',
 'D-30',
 'D-20',
 'DEALER_CITY',
 'D+6',
 'AKSHAYA_TRITIYA_DAYS',
 'X_CITY_CATEGORY',
 'MAHA_SHIVARATRI_DAYS',
 'REPUBLIC_DAY_DAYS',
 'ONAM_DAYS',
 'IGNITION_TYPE',
 'D0',
 'PARENT_DEALER_CODE',
 'BUDDHA_PURNIMA_DAYS',
 'D+1',
 'HARTALIK_TEEJ_DAYS',
 'D+7',
 'EID_UL_FITR_DAYS',
 'D-26',
 'NEW_YEAR_DAYS',
 'D-8',
 'D-17',
 'D-5',
 'D-4',
 'D-1',
 'D-12',
 'JANMASHTAMI_DAYS',
 'WHEEL_TYPE',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'D-29',
 'D-27',
 'ZONAL_OFFICE_NAME',
 'D-24',
 'D-3',
 'D+5',
 'D-25',
 'BRAKE_TYPE',
 'NAG_PANCHAMI_DAYS',
 'D+2',
 'MODEL_FAMILY',
 'LOHRI_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GURU_PURNIMA_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'D-18',
 'RAKSHA_BANDHAN_DAYS',
 'D-7',
 'D-16',
 'D-19',
 'D+4',
 'D-21',
 'VISHWAKA

In [32]:
static_covariates

['D-6',
 'D-23',
 'D-13',
 'HANUMAN_JAYANTI_DAYS',
 'COLOUR',
 'D-10',
 'JAGANNATH_RATHYATRA_DAYS',
 'D-14',
 'HOLIKA_DAHAN_DAYS',
 'D-9',
 'D-22',
 'HOLI_DAYS',
 'D-2',
 'D-11',
 'D-28',
 'D+3',
 'D-15',
 'GANGA_DUSSEHRA_DAYS',
 'D-30',
 'D-20',
 'DEALER_CITY',
 'D+6',
 'AKSHAYA_TRITIYA_DAYS',
 'X_CITY_CATEGORY',
 'MAHA_SHIVARATRI_DAYS',
 'REPUBLIC_DAY_DAYS',
 'ONAM_DAYS',
 'IGNITION_TYPE',
 'D0',
 'PARENT_DEALER_CODE',
 'BUDDHA_PURNIMA_DAYS',
 'D+1',
 'HARTALIK_TEEJ_DAYS',
 'D+7',
 'EID_UL_FITR_DAYS',
 'D-26',
 'NEW_YEAR_DAYS',
 'D-8',
 'D-17',
 'D-5',
 'D-4',
 'D-1',
 'D-12',
 'JANMASHTAMI_DAYS',
 'WHEEL_TYPE',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'D-29',
 'D-27',
 'D-24',
 'ZONAL_OFFICE_NAME',
 'D-3',
 'D+5',
 'D-25',
 'BRAKE_TYPE',
 'NAG_PANCHAMI_DAYS',
 'D+2',
 'MODEL_FAMILY',
 'LOHRI_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GURU_PURNIMA_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'D-18',
 'RAKSHA_BANDHAN_DAYS',
 'D-7',
 'D-16',
 'D-19',
 'D+4',
 'D-21',
 'VISHWAKARMA_PUJA_DAYS']

In [33]:
future_covariates

['FESTIVE_PHASE_II',
 'LAST_YEAR_CONTRIBUTION',
 'MARRIAGE_DAYS',
 'FESTIVE_PHASE_I',
 'PROP_FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'PROP_PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_III',
 'PROP_FESTIVE_PHASE_I']

In [34]:
#Step 1 - Preparing the lookback data for the model
lookback_data_pandas_df = prediction_df.loc[((prediction_df.index>='2025-02-01')&(prediction_df.index<='2026-05-01')),target_plus_static_cols]

#Step 2 - Preparing the lookahead data for the model
lookahead_data_pandas_df = prediction_df.loc[((prediction_df.index>='2025-02-01')&(prediction_df.index<='2026-11-01')),static_plus_future_cov]

#Step 3 - Creating the darts time-series object from lookback data for the model
lookback_data_darts_df = TimeSeries.from_group_dataframe(df=lookback_data_pandas_df,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')

#Step 4 - Creating the darts time-series object from lookahead data for the model 
lookahead_data_darts_df = TimeSeries.from_group_dataframe(
    df=lookahead_data_pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    static_cols=static_covariates, 
    value_cols=future_covariates,
    freq='MS'
)

In [35]:
scaled_temporal = future_covariates_scaler.transform(lookahead_data_darts_df)

final_scaled_lookahead_data = transformer.transform(scaled_temporal)

In [36]:
target_scaled_data = target_scaler.transform(lookback_data_darts_df)

final_scaled_lookback_data = transformer.transform(target_scaled_data)

In [37]:
forecast_series = loaded_model.predict(
    n=6, 
    series=final_scaled_lookback_data, 
    future_covariates=final_scaled_lookahead_data
)

print("Forecast generated successfully!")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast generated successfully!


In [38]:
actual_pred_series = target_scaler.inverse_transform(forecast_series)

In [39]:
# Build output DataFrame for Apr'26 to Jun'26 predictions
records = []

for forecast, source_series in zip(actual_pred_series, lookahead_data_darts_df):
    # val_list retains original static covariates — use it as the label source
    series_name = source_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    forecast_values = forecast.values().flatten()

    for month, pred in zip(months, forecast_values):
        records.append({
            'MONTH_OF_SALE'                   : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY'  : series_name,
            'PREDICTED_SALES'                  : round(float(pred), 2)
        })

df_final_output = pd.DataFrame(records)
df_final_output['MONTH_OF_SALE'] = pd.to_datetime(df_final_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_final_output = df_final_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_final_output.shape}')
print(f'Months       : {df_final_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_final_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_final_output.head(10)

Output shape : (216588, 3)
Months       : ['2026-06-01' '2026-07-01' '2026-08-01' '2026-09-01' '2026-10-01'
 '2026-11-01']
Series count : 36098


,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
0,2026-06-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,1.93
1,2026-07-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,1.36
2,2026-08-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.87
3,2026-09-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.56
4,2026-10-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,1.71
5,2026-11-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,1.36
6,2026-06-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,2.15
7,2026-07-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,1.61
8,2026-08-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,1.17
9,2026-09-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.83


In [40]:
df_final_output["MONTH_OF_SALE"] = pd.to_datetime(df_final_output["MONTH_OF_SALE"])

In [41]:
df_final_output["MONTH_NAME"] = df_final_output["MONTH_OF_SALE"].dt.strftime('%B')

In [42]:
df_final_output.groupby("MONTH_NAME",as_index=False).agg(TOTAL_MONTHLY_SALES=("PREDICTED_SALES",'sum'))

,MONTH_NAME,TOTAL_MONTHLY_SALES
0,August,318355.07
1,July,418686.37
2,June,537746.52
3,November,663827.34
4,October,871943.58
5,September,251872.30


In [43]:
df_final_output.to_csv(r"Final_output_diwali_features_unscaled_Huber_Loss_June_to_November.csv",index=False)

In [36]:
agg_output = df_final_output.groupby("MONTH_NAME",as_index=False).agg(TOTAL_MONTHLY_SALES=("PREDICTED_SALES",'sum'))

agg_output.to_csv(r"Aggregated_output.csv",index=False)

In [46]:
df_final_output.groupby("MONTH_NAME",as_index=False).agg(TOTAL_MONTHLY_SALES=("PREDICTED_SALES",'sum'))

,MONTH_NAME,TOTAL_MONTHLY_SALES
0,August,53696.27
1,July,55206.11
2,June,279580.96


In [47]:
df_final_output.to_csv(r"Final_output_validation_set.csv",index=False)

### Explainability

In [48]:
from darts.explainability import TFTExplainer

In [49]:
N = 100

explainer = TFTExplainer(
    loaded_model,
    background_series=final_scaled_lookback_data[:N],
    background_future_covariates=final_scaled_lookahead_data[:N],
)
result = explainer.explain()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  3.29it/s]


In [50]:
# CHECKPOINT 1 — look at this output, find the right accessor names
print([m for m in dir(result) if not m.startswith('_')])

['available_components', 'explained_components', 'feature_importances', 'get_attention', 'get_decoder_importance', 'get_encoder_importance', 'get_explanation', 'get_feature_importances', 'get_static_covariates_importance']


In [51]:
dec_imp = result.get_decoder_importance()   # <-- adjust if needed
enc_imp = result.get_encoder_importance()   # <-- adjust if needed

In [52]:
print(type(dec_imp))
if isinstance(dec_imp, list):
    import pandas as pd
    dec_imp = pd.concat(dec_imp, axis=0)
    enc_imp = pd.concat(enc_imp, axis=0)

<class 'list'>


In [53]:
dec_top = dec_imp.mean(axis=0).sort_values(ascending=False)
enc_top = enc_imp.mean(axis=0).sort_values(ascending=False)

In [54]:
dec_imp.to_csv('decoder_importance.csv')
enc_imp.to_csv('encoder_importance.csv')

In [59]:
future_covariates

['DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'NUM_FESTIVE_DAYS_MONTH',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'ONAM_DAYS',
 'PITRAPAKSHA_DAYS',
 'RAKSHA_BANDHAN_DAYS',
 'REPUBLIC_DAY_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'VISHWAKARMA_PUJA_DAYS',
 'MARRIAGE_DAYS',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'TOTAL_DAYS_FESTIVE_PHASE_I',
 'TOTAL_DAYS_FESTIVE_PHASE_II',
 'TOTAL_DAYS_FESTIVE_PHASE_III',
 'TOTAL_DAYS_PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_EVENT_FESTIVE_PHASE_I',
 'P

In [60]:
final_future_covariates = [i for i in future_covariates if i not in ['NON_ZERO_FLAG','PARENT_DEALER_CODE_MODEL_FAMILY']]
final_future_covariates

['DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'NUM_FESTIVE_DAYS_MONTH',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'ONAM_DAYS',
 'PITRAPAKSHA_DAYS',
 'RAKSHA_BANDHAN_DAYS',
 'REPUBLIC_DAY_DAYS',
 'VASANT_PANCHAMI_DAYS',
 'VISHWAKARMA_PUJA_DAYS',
 'MARRIAGE_DAYS',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'PITRU_PAKSH',
 'TOTAL_DAYS_FESTIVE_PHASE_I',
 'TOTAL_DAYS_FESTIVE_PHASE_II',
 'TOTAL_DAYS_FESTIVE_PHASE_III',
 'TOTAL_DAYS_PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_EVENT_FESTIVE_PHASE_I',
 'P

In [62]:
final_static_covariates = static_covariates + ['PARENT_DEALER_CODE']

In [ ]:
with open(r"C:\Users\G0004878\Desktop\TFT_Data\Temporal-Fusion-Transformers-Material\Documentation for Gourav\static_covariates.txt",'w') as f:
    for feature in final_static_covariates:
        f.write(f"{feature}\n")

TypeError: write() argument must be str, not list